In [ ]:
import tensorflow as tf

if tf.test.gpu_device_name():
    print("GPU found:", tf.test.gpu_device_name())
else:
    print("No GPU found")

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # Actuellement, la mémoire n'est pas allouée à l'avance, mais au besoin
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        # La mémoire du GPU doit être configurée avant l'initialisation des GPU
        print(e)

GPU found: /device:GPU:0
1 Physical GPUs, 1 Logical GPUs


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/pyronear/2_preprocessed_sequences_with_interpolation_v0.csv")
df["new_label"]= df["new_label"].fillna(0)

In [ ]:
from utils_dataset import create_classical_dataset
df_train_val, df_test = create_classical_dataset(df, test_size =1000, total_train_val_size=12000)

In [ ]:
df_train_val.head()

,Unnamed: 0,Extracted_Datetime,Dataset_prefix_group_id,Rel_Image_Path,Rel_Label_Path,Image_basename,Label_basename,Origin_dataset_name,Datetime_Str,Extension,...,img_width,has_label,yolo_bbox_xcenter,yolo_bbox_ycenter,yolo_bbox_width,yolo_bbox_height,new_label,pred,nb_detections,group
15,26,2023-05-23 17:33:31,DS_fp_pyronear_brison_1_group_101_1,pyronear_ds_03_2024/images/train/ADF_1320_2023...,pyronear_ds_03_2024/labels/train/ADF_1320_2023...,ADF_1320_2023_05_23T17_33_31.jpg,ADF_1320_2023_05_23T17_33_31.txt,df_pyronear_ds_03_2024_train,2023_05_23T17_33_31,jpg,...,1280,True,0.439091,0.331463,0.024703,0.037833,1.0,NaN,1.0,1
16,27,2023-05-23 17:34:31,DS_fp_pyronear_brison_1_group_101_1,pyronear_ds_03_2024/images/train/ADF_1320_2023...,pyronear_ds_03_2024/labels/train/ADF_1320_2023...,ADF_1320_2023_05_23T17_34_31.jpg,ADF_1320_2023_05_23T17_34_31.txt,df_pyronear_ds_03_2024_train,2023_05_23T17_34_31,jpg,...,1280,True,0.437414,0.336935,0.028057,0.024444,1.0,NaN,1.0,1
17,28,2023-05-23 17:35:00,DS_fp_pyronear_brison_1_group_101_1,pyronear_ds_03_2024/images/train/ADF_1320_2023...,pyronear_ds_03_2024/labels/train/ADF_1320_2023...,ADF_1320_2023_05_23T17_35_00.jpg,ADF_1320_2023_05_23T17_35_00.txt,df_pyronear_ds_03_2024_train,2023_05_23T17_35_00,jpg,...,1280,True,0.439128,0.336935,0.021901,0.024444,1.0,NaN,1.0,1
18,29,2023-05-23 17:35:30,DS_fp_pyronear_brison_1_group_101_1,pyronear_ds_03_2024/images/train/ADF_1320_2023...,pyronear_ds_03_2024/labels/train/ADF_1320_2023...,ADF_1320_2023_05_23T17_35_30.jpg,ADF_1320_2023_05_23T17_35_30.txt,df_pyronear_ds_03_2024_train,2023_05_23T17_35_30,jpg,...,1280,True,0.440495,0.336935,0.024635,0.022019,1.0,NaN,1.0,1
19,30,2023-05-23 17:36:00,DS_fp_pyronear_brison_1_group_101_1,pyronear_ds_03_2024/images/train/ADF_1320_2023...,pyronear_ds_03_2024/labels/train/ADF_1320_2023...,ADF_1320_2023_05_23T17_36_00.jpg,ADF_1320_2023_05_23T17_36_00.txt,df_pyronear_ds_03_2024_train,2023_05_23T17_36_00,jpg,...,1280,True,0.438065,0.336935,0.030859,0.031759,1.0,NaN,1.0,1


In [ ]:
from utils_dataset import prepare_sequences2

X_train_test, y_train_test = prepare_sequences2(df_train_val, data_dir = "/content/drive/MyDrive/pyronear/dataset_pyronear_yolo_lstm")

/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
Processing groups: 100%|██████████| 2400/2400 [00:00<00:00, 5205.62it/s]


Total processing time: 201.73 seconds


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_train_test, y_train_test, test_size=0.2, random_state=123)

In [ ]:
from utils_model import build_teacher_model

teacher_model = build_teacher_model()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
from utils_dataset import create_tensorflow_dataset

train_dataset, test_dataset = create_tensorflow_dataset(X_train, y_train, X_test, y_test, 16, 100)

In [ ]:
from utils_model import compile_and_fit

compile_and_fit(teacher_model, train_dataset, test_dataset, initial_epochs=0, fine_tune_epochs=1, fine_tune_layers=50, fine_tune_lr=1e-5)

Training with frozen layers
Fine-tuning with unfrozen layers
1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 235ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step
1/1 ━━

In [ ]:
df_train_val.to_csv("truc10_bbox2.csv")

In [ ]:
!cp frozen_metrics_log.csv /content/drive/MyDrive/pyronear/frozen12000_recall_firstalpha10_4_bbox2_1moreepoch.csv

In [ ]:
!cp unfrozen_metrics_log.csv /content/drive/MyDrive/pyronear/unfrozen12000_recall_firstalpha10r_4_bbox2_1moreepoch.csv

In [ ]:
teacher_model.save("/content/drive/MyDrive/pyronear/12000_recall_firstalpha10_bbox2_1moreepoch.h5")

In [ ]:
teacher_model.save_weights("/content/drive/MyDrive/pyronear/12000_recall_firstalpha10_bbox2_1moreepoch.weights.h5")